In [1]:
%run ../setup/01_config

Box(children=(Label(value='Catalog'), Text(value='dbr_dev')))

Box(children=(Label(value='Schema'), Text(value='bronze')))

In [2]:
from pyspark.sql import functions as F

In [3]:
dbutils.widgets.text(
    "table_name",
    "brz_menu_items",
    "Bronze Table Name"
)

TABLE_NAME = dbutils.widgets.get("table_name")

BRONZE_MENU_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

print(BRONZE_MENU_TABLE)

Box(children=(Label(value='Bronze Table Name'), Text(value='brz_menu_items')))

dbr_dev.bronze.brz_menu_items


In [4]:
df_menu = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(LANDING_MENU_PATH)
)

In [5]:
df_bronze_menu = (
    df_menu
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

In [6]:
(
    df_bronze_menu.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(BRONZE_MENU_TABLE)
)

In [7]:
spark.sql(f"DESCRIBE TABLE {BRONZE_MENU_TABLE}").show()

+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|        menu_item_id|      int|   NULL|
|           item_name|   string|   NULL|
|            category|   string|   NULL|
|               price|   double|   NULL|
|        _source_file|   string|   NULL|
|_ingestion_timestamp|timestamp|   NULL|
+--------------------+---------+-------+

